In [2]:
# list_events: List calendar events within a specified time range
# create_event: Create a new calendar event
# update_event: Update an existing calendar event
# delete_event: Delete a calendar event
# find_free_time: Find available time slots in the calendar
import os
import datetime
import os.path
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/calendar",
    "https://www.googleapis.com/auth/drive",
    "https://www.googleapis.com/auth/documents",
    "https://www.googleapis.com/auth/spreadsheets",
]


def main():
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())


if __name__ == "__main__":
    main()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=839260103192-fciii7ofgl4dmk0rcnh7u5lie89s63nf.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A42011%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdocuments+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets&state=KOrzKjwospJmDuiQzn76Muzon9h0t7&access_type=offline


/snap/core20/current/lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.29' not found (required by /lib/x86_64-linux-gnu/libproxy.so.1)
Failed to load module: /home/sahilsasane/snap/code/common/.cache/gio-modules/libgiolibproxy.so
Gtk-Message: 17:34:55.219: Not loading module "atk-bridge": The functionality is provided by GTK natively. Please try to not load it.


#### Calendar

In [ ]:
SCOPES = ["https://www.googleapis.com/auth/calendar"]
creds = Credentials.from_authorized_user_file("token.json", SCOPES)

In [3]:
calendarId = "aa32a783639e953b85e0a3d8f79abb74a9ae28ba9199c9a53f7598af6846697e@group.calendar.google.com"

In [71]:
def list_events(start_time, end_time):
    """Shows basic usage of the Google Calendar API.
    Prints the start and name of the next 10 events on the user's calendar.
    """
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    try:
        service = build("calendar", "v3", credentials=creds)
        now = datetime.datetime.now(tz=datetime.timezone.utc).isoformat()
        print("Getting the upcoming 10 events")
        events_result = (
            service.events()
            .list(
                calendarId=calendarId,
                timeMin=start_time,
                timeMax=end_time,
                maxResults=10,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        events = events_result.get("items", [])

        if not events:
            print("No upcoming events found.")

        # Prints the start and name of the next 10 events
        for event in events:
            start = event["start"].get("dateTime", event["start"].get("date"))
            print(start, event["summary"], "\nID: ", event["id"])

    except HttpError as error:
        print(f"An error occurred: {error}")


list_events(
    start_time="2025-10-01T00:00:00Z",
    end_time="2025-10-31T23:59:59Z",
)

Getting the upcoming 10 events
2025-10-01T10:00:00+05:30 Test Event 
ID:  09sv2rsbh4g52pqq8g1aa971fg


In [9]:
def list_events(start_time, end_time, calendarId):
    """Shows basic usage of the Google Calendar API.
    Returns the list of events on the user's calendar.
    """
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    try:
        service = build("calendar", "v3", credentials=creds)
        calendar_id = calendarId_base if calendarId != "primary" else calendarId
        print(calendar_id)
        events_result = (
            service.events()
            .list(
                calendarId=calendar_id,
                timeMin=start_time,
                timeMax=end_time,
                maxResults=10,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )
        print("hello")
        events = events_result.get("items", [])

        if not events:
            return {"message": "No upcoming events found.", "events": []}

        # Format events for return
        formatted_events = []
        for event in events:
            start = event["start"].get("dateTime", event["start"].get("date"))
            formatted_events.append(
                {"start": start, "summary": event["summary"], "id": event["id"]}
            )

        return {
            "message": f"Found {len(formatted_events)} events",
            "events": formatted_events,
        }

    except HttpError as error:
        return {"error": f"An error occurred: {error}"}

In [63]:
# Create Event
def create_event(start_time: str, end_time: str):
    """Create a new calendar event."""
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    try:
        service = build("calendar", "v3", credentials=creds)
        event = {
            "summary": "Test Event",
            "start": {"dateTime": start_time, "timeZone": "Asia/Kolkata"},
            "end": {"dateTime": end_time, "timeZone": "Asia/Kolkata"},
        }
        event = service.events().insert(calendarId=calendarId, body=event).execute()
        print(f"Event created: {event.get('htmlLink')}")
    except HttpError as error:
        print(f"An error occurred: {error}")
        event = None


create_event("2025-10-01T10:00:00", "2025-10-01T11:00:00")

Event created: https://www.google.com/calendar/event?eid=MDlzdjJyc2JoNGc1MnBxcThnMWFhOTcxZmcgYWEzMmE3ODM2MzllOTUzYjg1ZTBhM2Q4Zjc5YWJiNzRhOWFlMjhiYTkxOTljOWE1M2Y3NTk4YWY2ODQ2Njk3ZUBn


In [72]:
# Update Event
def update_event(event_id: str, start_time: str, end_time: str):
    """Update an existing calendar event."""
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("calendar", "v3", credentials=creds)
        service.events().update(
            calendarId=calendarId,
            eventId=event_id,
            body={
                "summary": "Updated Event",
                "start": {"dateTime": start_time, "timeZone": "Asia/Kolkata"},
                "end": {"dateTime": end_time, "timeZone": "Asia/Kolkata"},
            },
        ).execute()
        print(f"Event {event_id} updated successfully.")
    except HttpError as error:
        print(f"An error occurred: {error}")
        return None


update_event("09sv2rsbh4g52pqq8g1aa971fg", "2025-11-01T12:00:00", "2025-11-01T13:00:00")

Event 09sv2rsbh4g52pqq8g1aa971fg updated successfully.


In [73]:
# Delete Event
def delete_event(event_id: str):
    """Delete a calendar event."""
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("calendar", "v3", credentials=creds)
        service.events().delete(calendarId=calendarId, eventId=event_id).execute()
        print(f"Event {event_id} deleted successfully.")
    except HttpError as error:
        print(f"An error occurred: {error}")
        return None


delete_event("09sv2rsbh4g52pqq8g1aa971fg")

Event 09sv2rsbh4g52pqq8g1aa971fg deleted successfully.


#### Gmail

In [3]:
SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]
creds = Credentials.from_authorized_user_file("token.json", SCOPES)

In [ ]:
# Create Draft Email
import base64
from email.message import EmailMessage

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def gmail_create_draft():
    """Create and insert a draft email.
     Print the returned draft's message and id.
     Returns: Draft object, including draft id and message meta data.

    Load pre-authorized user credentials from the environment.
    TODO(developer) - See https://developers.google.com/identity
    for guides on implementing OAuth2 for the application.
    """
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        # create gmail api client
        service = build("gmail", "v1", credentials=creds)

        message = EmailMessage()

        message.set_content("This is automated draft mail")

        message["To"] = "gduser1@workspacesamples.dev"
        message["From"] = "gduser2@workspacesamples.dev"
        message["Subject"] = "Automated draft"

        # encoded message
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

        create_message = {"message": {"raw": encoded_message}}
        # pylint: disable=E1101
        draft = (
            service.users().drafts().create(userId="me", body=create_message).execute()
        )

        print(f"Draft id: {draft['id']}\nDraft message: {draft['message']}")

    except HttpError as error:
        print(f"An error occurred: {error}")
        draft = None

    return draft


if __name__ == "__main__":
    gmail_create_draft()

Draft id: r-9122070113505758020
Draft message: {'id': '197e12b08037aa85', 'threadId': '197e12b08037aa85', 'labelIds': ['DRAFT']}


In [ ]:
# Gmail Send Email

import base64
from email.message import EmailMessage

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def gmail_send_email():
    """Create and insert a draft email."""
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        service = build("gmail", "v1", credentials=creds)
        message = EmailMessage()

        message.set_content("This is automated draft mail")

        message["To"] = "gduser1@workspacesamples.dev"
        message["From"] = "gduser2@workspacesamples.dev"
        message["Subject"] = "Automated draft"

        # encoded message
        encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

        create_message = {"raw": encoded_message}
        # pylint: disable=E1101
        send_message = (
            service.users().messages().send(userId="me", body=create_message).execute()
        )
        print(f"Message Id: {send_message['id']}")
    except HttpError as error:
        print(f"An error occurred: {error}")
        send_message = None
    return send_message


if __name__ == "__main__":
    gmail_send_email()

Message Id: 197e12e4304e413b


In [ ]:
# Get Threads
import google.auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


def show_chatty_threads():
    """Display threads with long conversations(>= 3 messages)"""
    creds = None

    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open("token.json", "w") as token:
            token.write(creds.to_json())

    try:
        # create gmail api client
        service = build("gmail", "v1", credentials=creds)

        threads = (
            service.users().threads().list(userId="me").execute().get("threads", [])
        )
        print(f"Found {len(threads)} threads")
        print(threads)
        for thread in threads:
            tdata = (
                service.users().threads().get(userId="me", id=thread["id"]).execute()
            )
            nmsgs = len(tdata["messages"])

            # skip if <3 msgs in thread
            if nmsgs > 2:
                msg = tdata["messages"][0]["payload"]
                subject = ""
                for header in msg["headers"]:
                    if header["name"] == "Subject":
                        subject = header["value"]
                        break
                if subject:  # skip if no Subject line
                    print(f"- {subject}, {nmsgs}")
        return threads

    except HttpError as error:
        print(f"An error occurred: {error}")


if __name__ == "__main__":
    show_chatty_threads()

Found 100 threads
[{'id': '197e5d17a3c76169', 'snippet': 'Verify your Google Cloud account by Nov 05, 2025 Verify your Google Cloud account by Nov 05, 2025 6558-7085-3479 Google is required to verify certain Google Cloud accounts to protect against fraud and', 'historyId': '576136'}, {'id': '1975414f8e2428ec', 'snippet': 'Dear Sir/Madam, Greetings from Training &amp; Placement Cell, Pillai College of Engineering! ReBIT, a 100% owned subsidiary of RBI is looking to hire Fresher Pass outs of 2025 for below roles. About', 'historyId': '567691'}, {'id': '197517216f4cf8bf', 'snippet': 'You can now make complex workflows more easily ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏', 'historyId': '567592'}, {'id': '1971b7f4e9f0df3d', 'snippet': 'We&#39;re excited to announce the 2nd Pre-Placement Talk Session, following the enthusiastic response to our first session. The students&#39; posi

In [ ]:
GMAIL_DIR = "gmail"
import json


async def get_recent_meeting_emails() -> str:
    """Get recent emails that appear to be about meetings"""
    try:
        service = build("gmail", "v1", credentials=creds)
        results = (
            service.users()
            .messages()
            .list(
                userId="me",
                q='meeting OR "let\'s discuss" OR "schedule a call" OR "set up a meeting"',
                maxResults=5,
            )
            .execute()
        )
        path = os.path.join(GMAIL_DIR, "recent_meeting_emails.json")
        os.makedirs(path, exist_ok=True)
        file_path = os.path.join(path, "recent_meeting_emails.json")

        messages = results.get("messages", [])
        emails_data = {}

        for msg in messages:
            email = service.users().messages().get(userId="me", id=msg["id"]).execute()

            sender = ""
            date = ""
            subject = ""
            receiver = ""

            content = base64.urlsafe_b64decode(
                email["payload"]["parts"][0]["body"]["data"]
            ).decode("utf-8")
            for i in email["payload"]["headers"]:
                if i["name"] == "From":
                    sender = i["value"]
                if i["name"] == "Date":
                    date = i["value"]
                if i["name"] == "Subject":
                    subject = i["value"]
                if i["name"] == "To":
                    receiver = i["value"]

            emails_data[msg["id"]] = {
                "subject": subject,
                "sender": sender,
                "receiver": receiver,
                "date": date,
                "content": content,
            }
        with open(file_path, "w") as file:
            existing_data = []
            if os.path.exists(file_path):
                with open(file_path, "r") as existing_file:
                    try:
                        existing_data = json.load(existing_file)
                    except json.JSONDecodeError:
                        pass

            existing_data.append(emails_data)
            with open(file_path, "w") as file:
                json.dump(existing_data, file, indent=4)

        print(
            f"Found {len(emails_data)} recent meeting-related emails and stored them in {file_path}"
        )

    except Exception as e:
        return f"❌ Error fetching emails: {str(e)}"


def extract_recent_meeting_emails():
    """Extract recent meeting-related emails from Gmail."""
    try:
        path = os.path.join(GMAIL_DIR, "recent_meeting_emails.json")
        email_data = {}
        if os.path.exists(path):
            with open(path, "r") as file:
                email_data = json.load(file)
        else:
            return "No recent meeting emails found."

        if not email_data:
            return "No recent meeting emails found."

        emails = []
        for email_id, email_info in email_data.items():
            emails.append(
                {
                    "id": email_id,
                    "subject": email_info.get("subject", ""),
                    "sender": email_info.get("sender", ""),
                    "receiver": email_info.get("receiver", ""),
                    "date": email_info.get("date", ""),
                    "content": email_info.get("content", ""),
                }
            )
        return emails

    except Exception as e:
        return f"❌ Error fetching emails: {str(e)}"

In [27]:
import json
import base64

service = build("gmail", "v1", credentials=creds)
results = (
    service.users()
    .messages()
    .list(
        userId="me",
        q='meeting OR "let\'s discuss" OR "schedule a call" OR "set up a meeting"',
        maxResults=5,
    )
    .execute()
)

messages = results.get("messages", [])
emails_data = {}

for msg in messages:
    email = service.users().messages().get(userId="me", id=msg["id"]).execute()

    sender = ""
    date = ""
    subject = ""
    receiver = ""

    snippet = email.get("snippet")
    content = base64.urlsafe_b64decode(
        email["payload"]["parts"][0]["body"]["data"]
    ).decode("utf-8")
    for i in email["payload"]["headers"]:
        if i["name"] == "From":
            sender = i["value"]
        if i["name"] == "Date":
            date = i["value"]
        if i["name"] == "Subject":
            subject = i["value"]
        if i["name"] == "To":
            receiver = i["value"]

    emails_data[msg["id"]] = {
        "subject": subject,
        "sender": sender,
        "receiver": receiver,
        "date": date,
        "content": content,
    }

print(json.dumps(emails_data, indent=2))

{
  "197e639effcaaab3": {
    "subject": "Project Kickoff Meeting \u2013 Discuss Next Steps and Timeline",
    "sender": "Sahil <sahilsasane8@gmail.com>",
    "receiver": "sahilsasane21comp@student.mes.ac.in",
    "date": "Tue, 8 Jul 2025 00:20:34 +0530",
    "content": "Hi Team,\r\n\r\nI hope you\u2019re doing well. I\u2019d like to schedule a project kickoff meeting to\r\ndiscuss the next steps for the new product launch. We need to align on the\r\nproject timeline, key deliverables, and responsibilities.\r\n\r\nProposed topics:\r\n\r\n    Finalizing the project scope\r\n\r\n    Assigning roles and tasks\r\n\r\n    Setting key milestones and deadlines\r\n\r\n    Addressing any immediate concerns\r\n\r\nSuggested date and time: Friday at 3:00 PM. Please confirm if this works or\r\npropose an alternative.\r\n\r\nAttendees: alice@example.com, bob@example.com, charlie@example.com\r\n\r\nLet me know if there\u2019s anything else to prepare in advance.\r\n\r\nBest regards,\r\nDavid\r\n"
  

In [22]:
email = service.users().messages().get(userId="me", id=messages[0]["id"]).execute()
email

{'id': '197e639effcaaab3',
 'threadId': '197e639effcaaab3',
 'labelIds': ['IMPORTANT', 'CATEGORY_PERSONAL', 'INBOX'],
 'snippet': 'Hi Team, I hope you&#39;re doing well. I&#39;d like to schedule a project kickoff meeting to discuss the next steps for the new product launch. We need to align on the project timeline, key',
 'payload': {'partId': '',
  'mimeType': 'multipart/alternative',
  'filename': '',
  'headers': [{'name': 'Delivered-To',
    'value': 'sahilsasane21comp@student.mes.ac.in'},
   {'name': 'Received',
    'value': 'by 2002:a05:6022:539a:b0:6b:b4c1:b4f8 with SMTP id cl26csp7256058lab;        Mon, 7 Jul 2025 11:50:46 -0700 (PDT)'},
   {'name': 'X-Received',
    'value': 'by 2002:a05:6808:1b1e:b0:401:bb42:700c with SMTP id 5614622812f47-41114d724e7mr426351b6e.19.1751914245936;        Mon, 07 Jul 2025 11:50:45 -0700 (PDT)'},
   {'name': 'ARC-Seal',
    'value': 'i=1; a=rsa-sha256; t=1751914245; cv=none;        d=google.com; s=arc-20240605;        b=U3BRgAfR4Ijk7xpsVAQ7fzYBt

In [25]:
email["payload"]["parts"][0]["body"]["data"]

'SGkgVGVhbSwNCg0KSSBob3BlIHlvdeKAmXJlIGRvaW5nIHdlbGwuIEnigJlkIGxpa2UgdG8gc2NoZWR1bGUgYSBwcm9qZWN0IGtpY2tvZmYgbWVldGluZyB0bw0KZGlzY3VzcyB0aGUgbmV4dCBzdGVwcyBmb3IgdGhlIG5ldyBwcm9kdWN0IGxhdW5jaC4gV2UgbmVlZCB0byBhbGlnbiBvbiB0aGUNCnByb2plY3QgdGltZWxpbmUsIGtleSBkZWxpdmVyYWJsZXMsIGFuZCByZXNwb25zaWJpbGl0aWVzLg0KDQpQcm9wb3NlZCB0b3BpY3M6DQoNCiAgICBGaW5hbGl6aW5nIHRoZSBwcm9qZWN0IHNjb3BlDQoNCiAgICBBc3NpZ25pbmcgcm9sZXMgYW5kIHRhc2tzDQoNCiAgICBTZXR0aW5nIGtleSBtaWxlc3RvbmVzIGFuZCBkZWFkbGluZXMNCg0KICAgIEFkZHJlc3NpbmcgYW55IGltbWVkaWF0ZSBjb25jZXJucw0KDQpTdWdnZXN0ZWQgZGF0ZSBhbmQgdGltZTogRnJpZGF5IGF0IDM6MDAgUE0uIFBsZWFzZSBjb25maXJtIGlmIHRoaXMgd29ya3Mgb3INCnByb3Bvc2UgYW4gYWx0ZXJuYXRpdmUuDQoNCkF0dGVuZGVlczogYWxpY2VAZXhhbXBsZS5jb20sIGJvYkBleGFtcGxlLmNvbSwgY2hhcmxpZUBleGFtcGxlLmNvbQ0KDQpMZXQgbWUga25vdyBpZiB0aGVyZeKAmXMgYW55dGhpbmcgZWxzZSB0byBwcmVwYXJlIGluIGFkdmFuY2UuDQoNCkJlc3QgcmVnYXJkcywNCkRhdmlkDQo='